# Figures de présentation — Pipeline MSD Pancreas

Génère deux figures clés :
1. **Cascade pancréas → tumeur** : coupe axiale avec box pancréas inflatée (vert) et box tumeur (rouge)
2. **Stack 3 canaux** : coupes prev / curr / next côte à côte, puis le crop 3-canaux empilé

In [ ]:
# === Colab bootstrap ===
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys, subprocess
    REPO = '/content/INF8225_Projet'
    if not os.path.isdir(REPO):
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'temp',
            'https://github.com/Azcatchi17/INF8225_Projet.git', REPO])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from colab.setup import setup
    setup()
else:
    print('Local run')

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from skimage import io

from msd_implementation.pipelines.dino_medsam_gemini import config
from msd_implementation.pipelines.three_slice_context.slice_stack import (
    parse_slice, find_best_neighbor, stack_3slice_image, _load_grayscale,
)
from msd_implementation.pipelines.dino_medsam_cascade.cascade_detector import (
    detect_with_labels, _split_by_label, _inflate_box, _overlap_ratio,
    PANCREAS_LABEL, TUMOR_LABEL,
)

SAVE_DIR = Path('report/figures/msd')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

with open(config.MSD_TEST_JSON) as f:
    test_meta = json.load(f)

print(f'{len(test_meta["images"])} images test')

## Choix d'un cas tumor-present avec bonne détection

In [ ]:
# Trouver un bon exemple : tumeur visible, DINO détecte pancréas + tumeur
TEXT_PROMPT = 'pancreas . tumor .'
PANCREAS_THR = 0.3
TUMOR_THR = 0.01
MARGIN_PX = 20

best_example = None
best_score = 0

# On cherche parmi les images avec annotation tumeur
ann_by_img = {}
for ann in test_meta.get('annotations', []):
    ann_by_img.setdefault(ann['image_id'], []).append(ann)

tumor_images = [img for img in test_meta['images'] if img['id'] in ann_by_img]
print(f'Scanning {len(tumor_images)} tumor-present images...')

for img_info in tumor_images[:20]:
    img_path = str(config.MSD_ROOT / img_info['file_name'])
    boxes = detect_with_labels(img_path, text=TEXT_PROMPT, score_threshold=min(PANCREAS_THR, TUMOR_THR))
    pancreas = [b for b in boxes if b.label == PANCREAS_LABEL and b.score > PANCREAS_THR]
    tumors = [b for b in boxes if b.label == TUMOR_LABEL and b.score > TUMOR_THR]
    
    if pancreas and tumors:
        top_tumor = max(tumors, key=lambda b: b.score)
        if top_tumor.score > best_score:
            best_score = top_tumor.score
            best_example = {
                'path': img_path,
                'info': img_info,
                'pancreas': pancreas,
                'tumors': tumors,
            }
            print(f"  {img_info['file_name']}: pancreas={len(pancreas)}, tumors={len(tumors)}, best_score={top_tumor.score:.3f}")

assert best_example is not None, 'Aucun exemple trouvé avec pancréas + tumeur détectés'
print(f"\nExemple choisi: {best_example['info']['file_name']} (score={best_score:.3f})")

## Figure 1 — Cascade pancréas → tumeur

In [ ]:
img = np.array(Image.open(best_example['path']).convert('L'))
best_p = max(best_example['pancreas'], key=lambda b: b.score)
best_t = max(best_example['tumors'], key=lambda b: b.score)
p_inflated = _inflate_box(best_p.xyxy, MARGIN_PX)
overlap = _overlap_ratio(best_t.xyxy, p_inflated)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Panel 1 : détection brute ---
ax = axes[0]
ax.imshow(img, cmap='gray')
for b in best_example['pancreas']:
    x1, y1, x2, y2 = b.xyxy
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
        linewidth=1.5, edgecolor='lime', facecolor='none', linestyle='-', alpha=0.7)
    ax.add_patch(rect)
for b in best_example['tumors']:
    x1, y1, x2, y2 = b.xyxy
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
        linewidth=1.5, edgecolor='red', facecolor='none', linestyle='-', alpha=0.7)
    ax.add_patch(rect)
ax.set_title('(a) Détections DINO brutes', fontsize=11)
ax.axis('off')

# --- Panel 2 : pancréas inflaté + filtrage ---
ax = axes[1]
ax.imshow(img, cmap='gray')
x1, y1, x2, y2 = p_inflated
rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
    linewidth=2, edgecolor='lime', facecolor='lime', alpha=0.1, linestyle='--')
ax.add_patch(rect)
rect2 = patches.Rectangle((x1, y1), x2-x1, y2-y1,
    linewidth=2, edgecolor='lime', facecolor='none', linestyle='--')
ax.add_patch(rect2)
tx1, ty1, tx2, ty2 = best_t.xyxy
rect = patches.Rectangle((tx1, ty1), tx2-tx1, ty2-ty1,
    linewidth=2, edgecolor='red', facecolor='none')
ax.add_patch(rect)
ax.set_title(f'(b) ROI pancréas (+{MARGIN_PX}px) + tumeur', fontsize=11)
ax.axis('off')

# --- Panel 3 : masque GT superposé ---
ax = axes[2]
ax.imshow(img, cmap='gray')
# Charger le masque GT
mask_path = best_example['path'].replace('/images/', '/masks/')
if Path(mask_path).exists():
    mask = np.array(Image.open(mask_path))
    tumor_mask = (mask == 2).astype(float)
    ax.imshow(tumor_mask, cmap='Reds', alpha=0.4)
    pancreas_mask = (mask == 1).astype(float)
    ax.imshow(pancreas_mask, cmap='Greens', alpha=0.15)
rect = patches.Rectangle((tx1, ty1), tx2-tx1, ty2-ty1,
    linewidth=2, edgecolor='red', facecolor='none')
ax.add_patch(rect)
ax.set_title('(c) Vérité terrain + box sélectionnée', fontsize=11)
ax.axis('off')

plt.tight_layout()
fig.savefig(SAVE_DIR / 'cascade_detection.pdf', dpi=300, bbox_inches='tight')
fig.savefig(SAVE_DIR / 'cascade_detection.png', dpi=300, bbox_inches='tight')
print(f'Saved to {SAVE_DIR}/cascade_detection.{{pdf,png}}')
plt.show()

## Figure 2 — Stack 3 canaux (contexte inter-coupes)

In [ ]:
img_path = Path(best_example['path'])
parsed = parse_slice(img_path)
patient_id, slice_idx = parsed
print(f'Patient {patient_id}, slice {slice_idx}')

prev_path = find_best_neighbor(img_path, direction=-1)
next_path = find_best_neighbor(img_path, direction=+1)
print(f'prev: {prev_path}')
print(f'curr: {img_path}')
print(f'next: {next_path}')

curr = _load_grayscale(img_path)
prev = _load_grayscale(prev_path) if prev_path else curr
nxt = _load_grayscale(next_path) if next_path else curr

# Charger les masques correspondants
def load_mask(p):
    if p is None:
        return None
    mp = str(p).replace('/images/', '/masks/')
    return np.array(Image.open(mp)) if Path(mp).exists() else None

mask_prev = load_mask(prev_path)
mask_curr = load_mask(img_path)
mask_next = load_mask(next_path)

prev_info = parse_slice(prev_path) if prev_path else None
next_info = parse_slice(next_path) if next_path else None

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5),
                         gridspec_kw={'width_ratios': [1, 1, 1, 1.15]})

slices = [
    (prev, mask_prev, prev_info, 'prev'),
    (curr, mask_curr, (patient_id, slice_idx), 'curr'),
    (nxt, mask_next, next_info, 'next'),
]

# Panels 1-3 : trois coupes individuelles
for i, (sl, msk, info, label) in enumerate(slices):
    ax = axes[i]
    ax.imshow(sl, cmap='gray')
    if msk is not None:
        tumor_m = (msk == 2).astype(float)
        if tumor_m.any():
            ax.imshow(tumor_m, cmap='Reds', alpha=0.4)
    sid = f's{info[1]}' if info else '?'
    channel_colors = ['#4dabf7', '#ffd43b', '#ff6b6b']
    ax.set_title(f'({chr(97+i)}) {label} — {sid}', fontsize=11,
                 color=channel_colors[i], fontweight='bold')
    ax.axis('off')
    # Bordure colorée pour identifier le canal
    for spine in ax.spines.values():
        spine.set_edgecolor(channel_colors[i])
        spine.set_linewidth(3)
        spine.set_visible(True)

# Panel 4 : stack 3 canaux en pseudo-RGB
ax = axes[3]
stack = np.stack([prev, curr, nxt], axis=-1)
ax.imshow(stack)
ax.set_title('(d) Stack 3 canaux → ResNet', fontsize=11, fontweight='bold')
ax.axis('off')
for spine in ax.spines.values():
    spine.set_edgecolor('white')
    spine.set_linewidth(3)
    spine.set_visible(True)

# Flèches entre les panels
for i in range(2):
    fig.text((i+1)*0.235 + 0.015, 0.5, '→', fontsize=20, ha='center', va='center',
             color='gray', fontweight='bold')
fig.text(0.72, 0.5, '⇒', fontsize=24, ha='center', va='center',
         color='white', fontweight='bold')

plt.tight_layout()
fig.savefig(SAVE_DIR / 'three_slice_stack.pdf', dpi=300, bbox_inches='tight')
fig.savefig(SAVE_DIR / 'three_slice_stack.png', dpi=300, bbox_inches='tight')
print(f'Saved to {SAVE_DIR}/three_slice_stack.{{pdf,png}}')
plt.show()

## Figure 3 — Crop 3 canaux avec marge (ce que le ResNet voit)

In [ ]:
CROP_MARGIN = 30
tx1, ty1, tx2, ty2 = best_t.xyxy

fig, axes = plt.subplots(1, 5, figsize=(20, 4),
                         gridspec_kw={'width_ratios': [1, 1, 1, 0.15, 1.2]})

channel_colors = ['#4dabf7', '#ffd43b', '#ff6b6b']
channel_labels = ['prev', 'curr', 'next']
h, w = curr.shape[:2]

# Crop coordinates with margin
cx1 = max(0, int(tx1) - CROP_MARGIN)
cy1 = max(0, int(ty1) - CROP_MARGIN)
cx2 = min(w, int(tx2) + CROP_MARGIN)
cy2 = min(h, int(ty2) + CROP_MARGIN)

crops = [prev[cy1:cy2, cx1:cx2], curr[cy1:cy2, cx1:cx2], nxt[cy1:cy2, cx1:cx2]]

for i in range(3):
    ax = axes[i]
    ax.imshow(crops[i], cmap='gray')
    # Dessiner la box tumeur originale (sans marge) en rouge
    bx1 = int(tx1) - cx1
    by1 = int(ty1) - cy1
    bw = int(tx2 - tx1)
    bh = int(ty2 - ty1)
    rect = patches.Rectangle((bx1, by1), bw, bh,
        linewidth=1.5, edgecolor='red', facecolor='none', linestyle='--')
    ax.add_patch(rect)
    ax.set_title(f'{channel_labels[i]}', fontsize=11,
                 color=channel_colors[i], fontweight='bold')
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor(channel_colors[i])
        spine.set_linewidth(3)
        spine.set_visible(True)

# Séparateur
axes[3].axis('off')
axes[3].text(0.5, 0.5, '⇒', fontsize=28, ha='center', va='center',
             transform=axes[3].transAxes, color='gray', fontweight='bold')

# Stack RGB
ax = axes[4]
crop_stack = np.stack(crops, axis=-1)
ax.imshow(crop_stack)
rect = patches.Rectangle((bx1, by1), bw, bh,
    linewidth=2, edgecolor='white', facecolor='none', linestyle='--')
ax.add_patch(rect)
ax.set_title(f'Stack RGB (marge={CROP_MARGIN}px)', fontsize=11, fontweight='bold')
ax.axis('off')

plt.tight_layout()
fig.savefig(SAVE_DIR / 'three_slice_crop.pdf', dpi=300, bbox_inches='tight')
fig.savefig(SAVE_DIR / 'three_slice_crop.png', dpi=300, bbox_inches='tight')
print(f'Saved to {SAVE_DIR}/three_slice_crop.{{pdf,png}}')
plt.show()

In [ ]:
print('Figures générées :')
for p in sorted(SAVE_DIR.glob('*')):
    print(f'  {p} ({p.stat().st_size / 1024:.0f} KB)')